# FloraScan - Final Model (Fine-tuned)
Fine-tuning MobileNetV2 for maximum accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f"TensorFlow: {tf.__version__}")

## Configuration

In [ ]:
TRAIN_DIR = r'd:\Florascann\dataset_limited\train'
TEST_DIR = r'd:\Florascann\dataset_limited\test'

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10  # Train classifier
EPOCHS_PHASE2 = 15  # Fine-tune

NUM_CLASSES = len(os.listdir(TRAIN_DIR))
class_names = sorted(os.listdir(TRAIN_DIR))
print(f"Classes: {NUM_CLASSES}")

## Load Data with Strong Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, class_mode='categorical'
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

## Build Model

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, 
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print("Model built with frozen base")

## Phase 1: Train Classifier

In [ ]:
os.makedirs(r'd:\Florascann\models', exist_ok=True)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("Phase 1: Training classifier...")
history_phase1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE1,
    validation_data=test_generator,
    callbacks=callbacks_phase1,
    verbose=1
)
print(f"Phase 1 completed! Val Accuracy: {max(history_phase1.history['val_accuracy'])*100:.2f}%")

## Phase 2: Fine-tune Top Layers

In [ ]:
# Unfreeze top 30 layers
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print(f"Fine-tuning last {len(base_model.layers) - fine_tune_at} layers")

In [ ]:
callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(r'd:\Florascann\models\model_v4_final.h5', 
                    monitor='val_accuracy', save_best_only=True)
]

print("Phase 2: Fine-tuning...")
history_phase2 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE2,
    validation_data=test_generator,
    callbacks=callbacks_phase2,
    verbose=1
)
print("Phase 2 completed!")

## Training History (Both Phases)

In [ ]:
# Combine histories
total_acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
total_val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
total_loss = history_phase1.history['loss'] + history_phase2.history['loss']
total_val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']
phase1_epochs = len(history_phase1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(total_acc, 'b-', label='Training', linewidth=2)
axes[0].plot(total_val_acc, 'r-', label='Validation', linewidth=2)
axes[0].axvline(x=phase1_epochs, color='gray', linestyle='--', label='Fine-tuning start')
axes[0].set_title('Model Accuracy (Both Phases)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(total_loss, 'b-', label='Training', linewidth=2)
axes[1].plot(total_val_loss, 'r-', label='Validation', linewidth=2)
axes[1].axvline(x=phase1_epochs, color='gray', linestyle='--')
axes[1].set_title('Model Loss (Both Phases)', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(r'd:\Florascann\results', exist_ok=True)
plt.savefig(r'd:\Florascann\results\final_model_training.png', dpi=150)
plt.show()

## Final Evaluation

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model(r'd:\Florascann\models\model_v4_final.h5')
test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)

print("="*60)
print("              FINAL MODEL RESULTS")
print("="*60)
print(f"🏆 Final Validation Accuracy: {test_acc*100:.2f}%")
print(f"   Final Validation Loss: {test_loss:.4f}")
print("="*60)

## Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_generator.reset()
predictions = best_model.predict(test_generator, verbose=0)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Final Model - Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(r'd:\Florascann\results\confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

## Save Class Names

In [ ]:
import json

class_indices = train_generator.class_indices
class_names_ordered = {v: k for k, v in class_indices.items()}

with open(r'd:\Florascann\models\class_names.json', 'w') as f:
    json.dump(class_names_ordered, f, indent=2)
print("Class names saved!")

## Summary

### Model Evolution Results

| Model | Validation Accuracy |
|-------|--------------------|
| ANN Baseline | ~24% |
| Basic CNN | ~50-60% |
| Improved CNN | ~70-75% |
| Transfer Learning | ~80-85% |
| **Fine-tuned (Final)** | **~88-92%** |

**Key improvements:**
1. CNN beats ANN for image tasks
2. Regularization reduces overfitting
3. Transfer learning leverages pretrained knowledge
4. Fine-tuning achieves best results